In [1]:
!pip install pytorchvideo
import pytorchvideo
import torch 
import time
from torch import nn
import torchvision
import matplotlib.pyplot as plt 
import seaborn as sns 
import numpy as np 
import pandas as pd
import mediapipe as mp
from mediapipe.tasks import python 
from mediapipe.tasks.python import vision
# from mediapipe.tasks.python.vision import drawing_utils
import cv2

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


# Abordagens
Existe o entendimento do video, como uma sequencia de elementos os quais podemos ter dentro de um espaco-tempo determinado, de maneira sequencial 
e possuindo ou nao algum contexyto ou signficado previo. A partir disso, temos que um dos entendimentos que podemos ter da libras, seria o de usar apeas
a primeira e a ultima imagem nos sinais, com o intuito de terminar o significado de cada glifo

In [2]:
model_path = "./hand_landmarker.task"

HAND_CONNECTIONS = [
    # Polegar
    (0,1), (1,2), (2,3), (3,4),

    # Indicador
    (0,5), (5,6), (6,7), (7,8),

    # Médio
    (5,9), (9,10), (10,11), (11,12),

    # Anelar
    (9,13), (13,14), (14,15), (15,16),

    # Mindinho
    (13,17), (17,18), (18,19), (19,20),

    # Palma
    (0,17)
]


BaseOptions = mp.tasks.BaseOptions
HandLandmarker = mp.tasks.vision.HandLandmarker
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions
HandLandmarkerResult = mp.tasks.vision.HandLandmarkerResult
VisionRunningMode = mp.tasks.vision.RunningMode

# Create a hand landmarker instance with the live stream mode:
def print_result(result: HandLandmarkerResult, output_image: mp.Image, timestamp_ms: int):
    print('hand landmarker result: {}'.format(result))

options = HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=model_path),
    running_mode=VisionRunningMode.LIVE_STREAM,
    result_callback=print_result)
landmarker = HandLandmarker.create_from_options(options) 

I0000 00:00:1779129674.062997  131818 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1779129674.071696  131835 gl_context.cc:385] GL version: 3.2 (OpenGL ES 3.2 Mesa 25.1.9), renderer: Mesa Intel(R) Graphics (ADL GT2)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1779129674.116466  131821 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1779129674.138885  131825 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [3]:
class VideoProcessor:
    def __init__(self, model_path: str) -> None:
        self.capture = cv2.VideoCapture(0)
        
        # Initialize the Landmarker Options
        self.options = HandLandmarkerOptions(
            base_options=BaseOptions(model_asset_path=model_path),
            running_mode=VisionRunningMode.VIDEO,
            num_hands=2
        )
        # Create the landmarker instance
        self.landmarker = HandLandmarker.create_from_options(self.options) 

    def start(self):
        if not self.capture.isOpened():
            self.free_resources()
            return
        
        print("Press 'q' or 'ESC' to quit.")
        
        while self.capture.isOpened():
            success, frame = self.capture.read()
            frame = cv2.flip(frame, 1)
            h, w, _ = frame.shape
            if not success:
                break

            # 1. Convert BGR (OpenCV) to RGB (MediaPipe)
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            
            # 2. Convert to MediaPipe Image object
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
            # 3. Calculate timestamp in milliseconds
            frame_timestamp_ms = int(time.time() * 1000)
            
            # 4. Detect hand landmarks
            # In VIDEO mode, you MUST provide a timestamp
            detection_result = self.landmarker.detect_for_video(mp_image, frame_timestamp_ms)
            # 5. Handle Results (Printing landmarker coordinates as an example)
           
            if detection_result.hand_landmarks:
                for normalized_landmarks_list in detection_result.hand_landmarks:
                    # print(normalized_landmarks) # Isso daqui seria uma lista de landkarms
                    points = []
                    for landmark in normalized_landmarks_list:
                        x_pixel = int(landmark.x * w)
                        y_pixel = int(landmark.y * h)
                        points.append((x_pixel, y_pixel))
                        cv2.circle(
                            frame,
                            (x_pixel, y_pixel),
                            5,
                            (255, 255, 255),
                            -1
                        )
                    for start_idx, end_idx in HAND_CONNECTIONS:

                        x1, y1 = points[start_idx]
                        x2, y2 = points[end_idx]

                        cv2.line(
                            frame,
                            (x1, y1),
                            (x2, y2),
                            (255, 255, 255),
                            2
                        )
            # Display the resulting frameq
            cv2.imshow('Webcam Capture', frame)
            
            # Exit conditions
            key = cv2.waitKey(1) & 0xFF
            if key == ord('q') or key == 27:
                break
        self.capture.release()
        self.free_resources()

    def free_resources(self):
        self.capture.release()
        cv2.destroyAllWindows()

if __name__ == "__main__":
    # Ensure the .task file is in your directory
    processor = VideoProcessor(model_path="hand_landmarker.task")
    processor.start()

I0000 00:00:1779129674.284903  131836 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1779129674.288673  131851 gl_context.cc:385] GL version: 3.2 (OpenGL ES 3.2 Mesa 25.1.9), renderer: Mesa Intel(R) Graphics (ADL GT2)
W0000 00:00:1779129674.320143  131842 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1779129674.336090  131840 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Press 'q' or 'ESC' to quit.


W0000 00:00:1779129674.636669  131848 landmark_projection_calculator.cc:78] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.
qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in "/home/vmoura/.local/lib/python3.13/site-packages/cv2/qt/plugins"
QFont::fromString: Invalid description 'Noto Sans,10,-1,5,400,0,0,0,0,0,0,0,0,0,0,1'
QFont::fromString: Invalid description 'Noto Sans Mono,10,-1,5,400,0,0,0,0,0,0,0,0,0,0,1'
QFont::fromString: Invalid description 'Noto Sans,10,-1,5,400,0,0,0,0,0,0,0,0,0,0,1'
QFont::fromString: Invalid description 'Noto Sans,9,-1,5,400,0,0,0,0,0,0,0,0,0,0,1'
